# Data preparation ntc data from ENTSO-E transparency

source: https://transparency.entsoe.eu/

Creates the following parsed datasets

- Country to country NTCs ("NTC_"+year+"_monthly_entsoe.csv")

Settings in next window

In [1]:
#download files again (yes/no)?
download = "no"

#set year for data creation
year = '2017'

In [2]:
import pysftp
import sys
import os
import pandas as pd
import datetime as dt
import wget
import calendar

In [3]:
#ENTSO-E sftp settings
host = "sftp-transparency.entsoe.eu"
password = "qhfWzbuxRkmmKb+"                
username = "jonas.savelsberg@unibas.ch"                
port = '22'

In [4]:
dir_out = "../parsed_data/"

In [5]:
map_country = {
    'AL' : 'AL',
'AT' : 'AT',
'BA' : 'BA',
'BG' : 'BG',
'CH' : 'CH',
'CZ' : 'CZ',
'DE_50HzT' : 'DE',
'DE_TenneT_GER' : 'DE',
'DE_TransnetBW' : 'DE',
'DK' : 'DK',
'ES' : 'ES',
'FR' : 'FR',
'GB' : 'GB',
'GR' : 'GR',
'HR' : 'HR',
'HU' : 'HU',
'IE' : 'IE',
'IT' : 'IT',
'ME' : 'ME',
'MK' : 'MK',
'NL' : 'NL',
'NO' : 'NO',
'NO2' : 'NO',
'PL' : 'PL',
'PT' : 'PT',
'RO' : 'RO',
'RS' : 'RS',
'SI' : 'SI',
'SK' : 'SK',
'TR' : 'TR',
'UA' : 'UA'}

In [6]:
#Connect to ENTSO-E Transparency FTP
#for this to work, pysftp 0.2.8 is needed!
path = '/TP_export/'
path_local = os.path.join(os.pardir,'source_data/ENTSOE/') 

In [7]:
# show list of all available folders (uncomment last line if needed)
with pysftp.Connection(host=host, username=username, password=password) as sftp:
    print("Connection succesfully established.")
    files = sftp.listdir('/TP_export/')   
    #print(files)

Connection succesfully established.


In [8]:
#load file names from server
path_ntc = path+'ForecastedDayAheadTransferCapacities/'
path_ntc_local = path_local+'ntc/'
with pysftp.Connection(host=host, username=username, password=password) as sftp:
    print("Connection succesfully established.")
    # show list of files
    files = sftp.listdir(path_ntc)
    if year != "":
        files = [i for i in files if year in i]

Connection succesfully established.


In [9]:
#download aggregated generation data (ForecastedDayAheadTransferCapacities)
if download == "yes":
    with pysftp.Connection(host=host, username=username, password=password) as sftp:
        for file in files:
            sftp.get(path_ntc+file,path_ntc_local+file)
            print('Successfully downloaded file '+file)

In [10]:
#combine files to one data frame
df_ntc_in = pd.DataFrame()
for file in files:
    df_temp = pd.read_csv(path_ntc_local+file,
                          decimal=".",encoding="UTF-16LE",sep="\t")
    df_ntc_in = df_ntc_in.append(df_temp)
df_ntc_in = df_ntc_in.drop(columns={'Year','Month','Day','ResolutionCode','OutAreaCode','OutAreaTypeCode','OutAreaName','InAreaCode','InAreaTypeCode','InAreaName','UpdateTime'})
df_ntc_in.OutMapCode = df_ntc_in.OutMapCode.map(map_country)
df_ntc_in.InMapCode = df_ntc_in.InMapCode.map(map_country)
df_ntc_in.head()

,DateTime,OutMapCode,InMapCode,ForecastTransferCapacity
0,2017-10-01 02:00:00.000,RO,HU,500.0
1,2017-10-01 06:00:00.000,RO,HU,500.0
2,2017-10-01 10:00:00.000,RO,HU,500.0
3,2017-10-01 14:00:00.000,RO,HU,500.0
4,2017-10-01 18:00:00.000,RO,HU,500.0


In [11]:
df_ntc_in.InMapCode.unique()

array(['HU', 'RO', 'AT', 'RS', 'HR', 'SK', nan, 'UA', 'GB', 'FR', 'ES',
       'AL', 'GR', 'ME', 'CH', 'BA', 'SI', 'BG', 'NL', 'IT', 'MK', 'CZ',
       'PL', 'DE', 'DK', 'NO', 'PT', 'TR', 'IE'], dtype=object)

In [12]:
df_ntc = df_ntc_in.groupby(['DateTime','OutMapCode','InMapCode']).sum().reset_index()
df_ntc = df_ntc.rename(columns={'DateTime':'date','OutMapCode':'from','InMapCode':'to','ForecastTransferCapacity':'ntc'})
df_ntc.head()

,date,from,to,ntc
0,2017-01-01 00:00:00.000,AL,GR,500.0
1,2017-01-01 00:00:00.000,AL,RS,500.0
2,2017-01-01 00:00:00.000,AT,CH,850.0
3,2017-01-01 00:00:00.000,AT,CZ,700.0
4,2017-01-01 00:00:00.000,AT,HU,600.0


In [13]:
df_ntc.to_csv(dir_out + "ntc_"+year+"_weekly_entsoe.csv", index=False)